# Distribution Reconstruction

## Goal

Convert GDP threshold probabilities into an implied market distribution.

In [ ]:
import pandas as pd
import numpy as np

from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR
from src.distribution import (
    build_market_distribution_panel,
    enforce_monotonic_survival_curve,
    implied_mean_from_buckets,
    implied_median_from_buckets,
    implied_variance_from_buckets,
    threshold_to_bucket_probabilities,
)

## Load Kalshi threshold probabilities

In [ ]:
raw_path = RAW_DATA_DIR / "kalshi_gdp_q2_2026_thresholds.csv"
threshold_df = pd.read_csv(raw_path)

# The raw pull is a current snapshot. Use the raw file timestamp as the panel date.
threshold_df["date"] = pd.to_datetime(raw_path.stat().st_mtime, unit="s").date().isoformat()
threshold_df["quarter"] = "2026Q2"
threshold_df["platform"] = "kalshi"

if threshold_df["threshold"].notna().sum() == 0:
    raise ValueError("Threshold count is zero.")

threshold_df[["date", "quarter", "platform", "ticker", "threshold", "prob_above"]]

## Enforce monotonic survival curve

In [ ]:
clean_df = enforce_monotonic_survival_curve(threshold_df)

for _, group in clean_df.sort_values("threshold").groupby(["date", "quarter", "platform"]):
    if np.any(np.diff(group["clean_prob_above"].to_numpy(dtype=float)) > 1e-12):
        raise ValueError("Clean probabilities are not monotonic non-increasing.")

clean_df[["threshold", "prob_above", "clean_prob_above", "monotonicity_error"]]

## Convert thresholds to buckets

In [ ]:
bucket_df = threshold_to_bucket_probabilities(clean_df)

if bucket_df.empty or (bucket_df["bucket_prob"] < -1e-12).any():
    raise ValueError("Bucket probabilities are empty or negative after cleaning.")

bucket_sums = bucket_df.groupby(["date", "quarter", "platform"])["bucket_prob"].sum()
if not np.allclose(bucket_sums.to_numpy(dtype=float), 1.0, atol=1e-6):
    raise ValueError(f"Bucket probabilities do not sum close to 1: {bucket_sums.to_dict()}")

bucket_df

## Estimate implied mean, median, variance

In [ ]:
mean_df = implied_mean_from_buckets(bucket_df)
median_df = implied_median_from_buckets(bucket_df)
variance_df = implied_variance_from_buckets(bucket_df)

metrics_df = mean_df.merge(median_df, on=["date", "quarter", "platform"]).merge(
    variance_df[["date", "quarter", "platform", "market_std_gdp"]],
    on=["date", "quarter", "platform"],
)

if metrics_df[["market_mean_gdp", "market_median_gdp", "market_std_gdp"]].isna().any().any():
    raise ValueError("Implied mean, median, or standard deviation is null.")

metrics_df

## Save market distribution panel

In [ ]:
panel = build_market_distribution_panel(threshold_df)
processed_path = PROCESSED_DATA_DIR / "kalshi_q2_2026_distribution_panel.csv"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
panel.to_csv(processed_path, index=False)
panel